# XGBoost and AdaBoost Models for Heat Risk Prediction

This notebook implements XGBoost and AdaBoost models for both regression (forecast heat index) and classification (predict heat risk) tasks, with comparison between the two approaches.

## Tasks:
1. Load train/validation/test splits
2. TODO: Feature preparation (reuse from Random Forest approach)
3. TODO: Build XGBoost regressor
4. TODO: Build XGBoost classifier
5. TODO: Build AdaBoost regressor
6. TODO: Build AdaBoost classifier
7. TODO: Hyperparameter tuning for both models
8. TODO: Evaluate and compare models
9. TODO: Feature importance analysis
10. TODO: Save models and predictions


## Step 1: Mount Google Drive and Import Libraries


In [ ]:
# Mount Google Drive and set repo path (Colab)
import os
repo_path = '/content/drive/MyDrive/SHADE-ML-Team'
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    if not os.path.exists(repo_path):
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        # Optionally clone if missing:
        # !git clone https://github.com/HrudithL/SHADE-ML-Team.git /content/drive/MyDrive/SHADE-ML-Team
    os.chdir(repo_path)
    print(f"Using repo at: {os.getcwd()}")
except ImportError:
    print("Running locally; ensure CWD is repo root")
    print(f"Current directory: {os.getcwd()}")



In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import AdaBoostRegressor, AdaBoostClassifier
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"XGBoost version: {xgb.__version__}")

# Set plotting style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('default')
sns.set_palette("husl")


## Step 2: Load Data Splits


In [ ]:
# Load train/validation/test splits
train_data = pd.read_csv('data/processed/train_data.csv')
val_data = pd.read_csv('data/processed/validation_data.csv')
test_data = pd.read_csv('data/processed/test_data.csv')

print("✓ Data loaded successfully!")
print(f"Train shape: {train_data.shape}")
print(f"Validation shape: {val_data.shape}")
print(f"Test shape: {test_data.shape}")
print(f"\nColumns: {train_data.columns.tolist()}")


## Step 3: Feature Preparation and Selection

Prepare features similar to Random Forest approach. **TODO**: Select which features to include.


In [ ]:
# TODO: SELECT FEATURES FOR XGBOOST/ADABOOST
# Review Random Forest notebook to see which features were used
# Decide which features to include (temperature, humidity, time-based, etc.)
# Consider feature engineering (creating new features from existing ones)

print("=" * 80)
print("FEATURE PREPARATION AND SELECTION")
print("=" * 80)

# Select features (you can reuse the same feature selection as Random Forest)
# TODO: Add/remove features based on your analysis and Random Forest results
feature_cols = [
    # Temperature features
    'TempHighF', 'TempLowF', 'TempAvgF',
    # Humidity and dew point
    'HumidityAvgPercent', 'DewPointAvgF',
    # Weather features
    'HeatIndex', 'sealevelpressure', 'VisibilityAvgMiles',
    # Time-based features
    'Year', 'Month', 'Day', 'DayOfYear',
    # TODO: Add derived features if available (e.g., 'DailyTempRange', 'TempAvg_7day', 'HumidityAvg_7day')
    # TODO: Add other relevant features based on domain knowledge
]

# Check which features exist
available_features = [col for col in feature_cols if col in train_data.columns]
missing_features = [col for col in feature_cols if col not in train_data.columns]

if missing_features:
    print(f"⚠ WARNING: Missing features: {missing_features}")
    feature_cols = available_features

print(f"\n✓ Selected {len(feature_cols)} features")
print(f"Features: {feature_cols}")

# Handle categorical variables (Label Encoding)
# TODO: Add other categorical columns if needed (e.g., 'Location', 'WeatherType')
categorical_cols = ['Season']  # TODO: Modify this list based on your data
existing_categorical = [col for col in categorical_cols if col in train_data.columns]

if existing_categorical:
    print(f"\nEncoding categorical variables: {existing_categorical}")
    label_encoders = {}
    
    for col in existing_categorical:
        le = LabelEncoder()
        train_data[f'{col}_encoded'] = le.fit_transform(train_data[col].astype(str))
        val_data[f'{col}_encoded'] = le.transform(val_data[col].astype(str))
        test_data[f'{col}_encoded'] = le.transform(test_data[col].astype(str))
        label_encoders[col] = le
    
    feature_cols.extend([f'{col}_encoded' for col in existing_categorical])
    feature_cols = [col for col in feature_cols if col not in categorical_cols]
    print(f"✓ Categorical variables encoded")

# Prepare features and targets
X_train = train_data[feature_cols].copy()
X_val = val_data[feature_cols].copy()
X_test = test_data[feature_cols].copy()

# Regression targets
y_train_reg = train_data['heat_index_next_day'].copy()
y_val_reg = val_data['heat_index_next_day'].copy()
y_test_reg = test_data['heat_index_next_day'].copy()

# Classification targets
y_train_cls = train_data['heat_risk'].copy()
y_val_cls = val_data['heat_risk'].copy()
y_test_cls = test_data['heat_risk'].copy()

# Handle NaN values in regression target
train_mask = ~y_train_reg.isnull()
val_mask = ~y_val_reg.isnull()
test_mask = ~y_test_reg.isnull()

X_train = X_train[train_mask].copy()
y_train_reg = y_train_reg[train_mask].copy()
y_train_cls = y_train_cls[train_mask].copy()

X_val = X_val[val_mask].copy()
y_val_reg = y_val_reg[val_mask].copy()
y_val_cls = y_val_cls[val_mask].copy()

X_test = X_test[test_mask].copy()
y_test_reg = y_test_reg[test_mask].copy()
y_test_cls = y_test_cls[test_mask].copy()

print(f"\nFinal feature shape: {X_train.shape}")
print(f"Regression target - Mean: {y_train_reg.mean():.2f}°F, Std: {y_train_reg.std():.2f}°F")
print(f"Classification target - Class 1: {(y_train_cls == 1).mean()*100:.2f}%")


## Step 4: Build XGBoost Regressor

Build and train XGBoost regressor. **TODO**: Adjust hyperparameters based on your experiments.


In [ ]:
# TODO: BUILD XGBOOST REGRESSOR
# Implement the XGBoost regressor: initialize, train, predict, and evaluate

print("=" * 80)
print("BUILDING XGBOOST REGRESSOR")
print("=" * 80)

# TODO: Set XGBoost hyperparameters
# Research appropriate values for regression tasks
# Consider: n_estimators, max_depth, learning_rate, subsample, colsample_bytree
xgb_n_estimators = None  # TODO: Set value (e.g., 100, 200, 300)
xgb_max_depth = None     # TODO: Set value (e.g., 3, 6, 9)
xgb_learning_rate = None # TODO: Set value (e.g., 0.01, 0.1, 0.3)
xgb_subsample = None      # TODO: Set value (e.g., 0.6, 0.8, 1.0)
xgb_colsample_bytree = None  # TODO: Set value (e.g., 0.6, 0.8, 1.0)

# TODO: Initialize XGBoost regressor
# Use xgb.XGBRegressor with the hyperparameters above
# Don't forget: random_state=42, n_jobs=-1, verbosity=0
# xgb_regressor = xgb.XGBRegressor(...)

# TODO: Train the model
# Fit the regressor on X_train and y_train_reg
# xgb_regressor.fit(...)

# TODO: Make predictions
# Predict on train, validation, and test sets
# y_train_pred_xgb_reg = ...
# y_val_pred_xgb_reg = ...
# y_test_pred_xgb_reg = ...

# TODO: Evaluate model performance
# Calculate RMSE and R² for train, validation, and test sets
# Use: np.sqrt(mean_squared_error(...)) for RMSE
# Use: r2_score(...) for R²
# train_rmse = ...
# val_rmse = ...
# test_rmse = ...
# train_r2 = ...
# val_r2 = ...
# test_r2 = ...

# TODO: Print evaluation results
# print("\n" + "=" * 80)
# print("XGBOOST REGRESSION PERFORMANCE")
# print("=" * 80)
# print(f"Train - RMSE: {train_rmse:.2f}, R²: {train_r2:.4f}")
# print(f"Val   - RMSE: {val_rmse:.2f}, R²: {val_r2:.4f}")
# print(f"Test  - RMSE: {test_rmse:.2f}, R²: {test_r2:.4f}")


In [ ]:
# TODO: BUILD XGBOOST CLASSIFIER
# Implement the XGBoost classifier: check class balance, initialize, train, predict, and evaluate

print("=" * 80)
print("BUILDING XGBOOST CLASSIFIER")
print("=" * 80)

# TODO: Check class balance
# Calculate class distribution and determine if classes are imbalanced
# class_balance = ...
# pos_weight = ...  # Calculate based on class distribution for scale_pos_weight
# Print class distribution information

# TODO: Set XGBoost classifier hyperparameters
# Reuse or adjust hyperparameters from regressor
# Set scale_pos_weight based on class balance
# xgb_scale_pos_weight = ...

# TODO: Initialize XGBoost classifier
# Use xgb.XGBClassifier with appropriate hyperparameters
# Include scale_pos_weight for imbalanced data handling
# Set eval_metric (e.g., 'logloss', 'auc', 'error')
# xgb_classifier = xgb.XGBClassifier(...)

# TODO: Train the classifier
# Fit on X_train and y_train_cls
# xgb_classifier.fit(...)

# TODO: Make predictions
# Predict classes on train, validation, and test sets
# y_train_pred_xgb_cls = ...
# y_val_pred_xgb_cls = ...
# y_test_pred_xgb_cls = ...

# TODO: Evaluate classification performance
# Calculate accuracy and F1-score for all sets
# Use: accuracy_score(...) and f1_score(..., zero_division=0)
# train_acc = ...
# val_acc = ...
# test_acc = ...
# train_f1 = ...
# val_f1 = ...
# test_f1 = ...

# TODO: Print evaluation results
# print("\n" + "=" * 80)
# print("XGBOOST CLASSIFICATION PERFORMANCE")
# print("=" * 80)
# print(f"Train - Accuracy: {train_acc:.4f}, F1: {train_f1:.4f}")
# print(f"Val   - Accuracy: {val_acc:.4f}, F1: {val_f1:.4f}")
# print(f"Test  - Accuracy: {test_acc:.4f}, F1: {test_f1:.4f}")


## Step 6: Build AdaBoost Regressor

Build and train AdaBoost regressor. **TODO**: Adjust hyperparameters (n_estimators, learning_rate).


In [ ]:
# TODO: BUILD ADABOOST REGRESSOR
# Implement the AdaBoost regressor: set hyperparameters, initialize, train, predict, and evaluate

print("=" * 80)
print("BUILDING ADABOOST REGRESSOR")
print("=" * 80)

# TODO: Set AdaBoost hyperparameters
# Research appropriate values for AdaBoost regression
# Consider: n_estimators, learning_rate, base estimator max_depth
ada_n_estimators = None  # TODO: Set value (e.g., 50, 100, 200)
ada_learning_rate = None  # TODO: Set value (e.g., 0.5, 1.0, 1.5)
base_max_depth = None  # TODO: Set value for base estimator depth (e.g., 1, 3, 5)

# TODO: Import DecisionTreeRegressor if needed
# from sklearn.tree import DecisionTreeRegressor

# TODO: Initialize AdaBoost regressor
# Use AdaBoostRegressor with DecisionTreeRegressor as base estimator
# Set the base estimator with max_depth parameter
# ada_regressor = AdaBoostRegressor(
#     estimator=DecisionTreeRegressor(max_depth=base_max_depth, random_state=42),
#     n_estimators=ada_n_estimators,
#     learning_rate=ada_learning_rate,
#     random_state=42
# )

# TODO: Train the model
# Fit on X_train and y_train_reg
# ada_regressor.fit(...)

# TODO: Make predictions
# Predict on train, validation, and test sets
# y_train_pred_ada_reg = ...
# y_val_pred_ada_reg = ...
# y_test_pred_ada_reg = ...

# TODO: Evaluate model performance
# Calculate RMSE and R² for all sets
# train_rmse = ...
# val_rmse = ...
# test_rmse = ...
# train_r2 = ...
# val_r2 = ...
# test_r2 = ...

# TODO: Print evaluation results
# print("\n" + "=" * 80)
# print("ADABOOST REGRESSION PERFORMANCE")
# print("=" * 80)
# print(f"Train - RMSE: {train_rmse:.2f}, R²: {train_r2:.4f}")
# print(f"Val   - RMSE: {val_rmse:.2f}, R²: {val_r2:.4f}")
# print(f"Test  - RMSE: {test_rmse:.2f}, R²: {test_r2:.4f}")


## Step 7: Build AdaBoost Classifier

Build and train AdaBoost classifier. **TODO**: Adjust hyperparameters and consider class_weight.


In [ ]:
# TODO: BUILD ADABOOST CLASSIFIER
# Implement the AdaBoost classifier: set hyperparameters, initialize, train, predict, and evaluate

print("=" * 80)
print("BUILDING ADABOOST CLASSIFIER")
print("=" * 80)

# TODO: Set AdaBoost classifier hyperparameters
# Consider adjusting base estimator depth for classification
# Reuse n_estimators and learning_rate from regressor or adjust
base_cls_max_depth = None  # TODO: Set value (e.g., 1, 3, 5)

# TODO: Import DecisionTreeClassifier if needed
# from sklearn.tree import DecisionTreeClassifier

# TODO: Initialize AdaBoost classifier
# Use AdaBoostClassifier with DecisionTreeClassifier as base estimator
# Consider class_weight='balanced' for imbalanced data
# ada_classifier = AdaBoostClassifier(
#     estimator=DecisionTreeClassifier(max_depth=base_cls_max_depth, random_state=42, class_weight='balanced'),
#     n_estimators=ada_n_estimators,
#     learning_rate=ada_learning_rate,
#     random_state=42
# )

# TODO: Train the classifier
# Fit on X_train and y_train_cls
# ada_classifier.fit(...)

# TODO: Make predictions
# Predict classes on train, validation, and test sets
# y_train_pred_ada_cls = ...
# y_val_pred_ada_cls = ...
# y_test_pred_ada_cls = ...

# TODO: Evaluate classification performance
# Calculate accuracy and F1-score for all sets
# train_acc = ...
# val_acc = ...
# test_acc = ...
# train_f1 = ...
# val_f1 = ...
# test_f1 = ...

# TODO: Print evaluation results
# print("\n" + "=" * 80)
# print("ADABOOST CLASSIFICATION PERFORMANCE")
# print("=" * 80)
# print(f"Train - Accuracy: {train_acc:.4f}, F1: {train_f1:.4f}")
# print(f"Val   - Accuracy: {val_acc:.4f}, F1: {val_f1:.4f}")
# print(f"Test  - Accuracy: {test_acc:.4f}, F1: {test_f1:.4f}")


## Step 8: Hyperparameter Tuning

Tune hyperparameters for both XGBoost and AdaBoost models. **TODO**: Enable tuning and adjust parameter grids.


In [ ]:
# TODO: DECIDE WHETHER TO PERFORM HYPERPARAMETER TUNING
# Set to True if you want to tune hyperparameters (takes longer)
perform_tuning = False  # TODO: Set to True to enable hyperparameter tuning

if perform_tuning:
    print("=" * 80)
    print("HYPERPARAMETER TUNING")
    print("=" * 80)
    
    # TODO: ADJUST PARAMETER GRIDS
    # Consider computational cost - start with smaller grids
    
    # XGBoost regression parameter grid
    param_grid_xgb_reg = {
        'n_estimators': [100, 200],  # TODO: Add more values if time permits
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1],
        'subsample': [0.8, 1.0]
    }
    
    print("\nTuning XGBoost Regression Model...")
    grid_search_xgb_reg = GridSearchCV(
        estimator=xgb.XGBRegressor(random_state=42, n_jobs=-1),
        param_grid=param_grid_xgb_reg,
        cv=5,  # TODO: Adjust CV folds if needed
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search_xgb_reg.fit(X_train, y_train_reg)
    print(f"Best parameters: {grid_search_xgb_reg.best_params_}")
    xgb_regressor = grid_search_xgb_reg.best_estimator_
    
    # XGBoost classification parameter grid
    param_grid_xgb_cls = {
        'n_estimators': [100, 200],
        'max_depth': [3, 6],
        'learning_rate': [0.01, 0.1],
        'scale_pos_weight': [pos_weight, pos_weight * 1.5]  # TODO: Adjust based on class balance
    }
    
    print("\nTuning XGBoost Classification Model...")
    grid_search_xgb_cls = GridSearchCV(
        estimator=xgb.XGBClassifier(random_state=42, n_jobs=-1),
        param_grid=param_grid_xgb_cls,
        cv=5,
        scoring='f1',  # TODO: Consider 'roc_auc' or 'accuracy'
        n_jobs=-1,
        verbose=1
    )
    
    grid_search_xgb_cls.fit(X_train, y_train_cls)
    print(f"Best parameters: {grid_search_xgb_cls.best_params_}")
    xgb_classifier = grid_search_xgb_cls.best_estimator_
    
    # AdaBoost parameter grids (simpler)
    param_grid_ada_reg = {
        'n_estimators': [50, 100],
        'learning_rate': [0.5, 1.0, 1.5]
        # TODO: Consider adding 'estimator__max_depth': [1, 3, 5] to tune base estimator
    }
    
    print("\nTuning AdaBoost Regression Model...")
    grid_search_ada_reg = GridSearchCV(
        estimator=AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=3, random_state=42), random_state=42),
        param_grid=param_grid_ada_reg,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search_ada_reg.fit(X_train, y_train_reg)
    print(f"Best parameters: {grid_search_ada_reg.best_params_}")
    ada_regressor = grid_search_ada_reg.best_estimator_
    
    print("\n✓ Hyperparameter tuning complete!")
else:
    print("=" * 80)
    print("SKIPPING HYPERPARAMETER TUNING")
    print("=" * 80)
    print("Set perform_tuning = True to enable hyperparameter tuning")
    print("Using initial models with default hyperparameters")


## Step 9: Evaluate and Compare Models

Comprehensive evaluation and comparison of XGBoost vs AdaBoost models.


In [ ]:
# TODO: COMPREHENSIVE MODEL EVALUATION AND COMPARISON
# Compare XGBoost vs AdaBoost for both regression and classification
# Create visualizations and comparison tables

print("=" * 80)
print("COMPREHENSIVE MODEL EVALUATION AND COMPARISON")
print("=" * 80)

# TODO: Refresh predictions for all models
# Make predictions on train, validation, and test sets for all four models
# y_train_pred_xgb_reg = ...
# y_val_pred_xgb_reg = ...
# y_test_pred_xgb_reg = ...

# y_train_pred_xgb_cls = ...
# y_val_pred_xgb_cls = ...
# y_test_pred_xgb_cls = ...

# y_train_pred_ada_reg = ...
# y_val_pred_ada_reg = ...
# y_test_pred_ada_reg = ...

# y_train_pred_ada_cls = ...
# y_val_pred_ada_cls = ...
# y_test_pred_ada_cls = ...

# TODO: Regression comparison
# Calculate metrics for both models on test set
print("\n" + "=" * 80)
print("REGRESSION COMPARISON (Test Set)")
print("=" * 80)

# TODO: Calculate test set metrics for both regression models
# xgb_test_rmse = ...
# ada_test_rmse = ...
# xgb_test_r2 = ...
# ada_test_r2 = ...

# TODO: Create comparison DataFrame
# comparison_reg = pd.DataFrame({
#     'Model': ['XGBoost', 'AdaBoost'],
#     'RMSE': [xgb_test_rmse, ada_test_rmse],
#     'R²': [xgb_test_r2, ada_test_r2]
# })
# print(comparison_reg.to_string(index=False))

# TODO: Classification comparison
# Calculate accuracy, F1, precision, recall for both models
print("\n" + "=" * 80)
print("CLASSIFICATION COMPARISON (Test Set)")
print("=" * 80)

# TODO: Calculate test set metrics
# xgb_test_acc = ...
# ada_test_acc = ...
# xgb_test_f1 = ...
# ada_test_f1 = ...

# TODO: Create comparison DataFrame
# comparison_cls = pd.DataFrame({
#     'Model': ['XGBoost', 'AdaBoost'],
#     'Accuracy': [xgb_test_acc, ada_test_acc],
#     'F1-Score': [xgb_test_f1, ada_test_f1]
# })
# print(comparison_cls.to_string(index=False))

# TODO: Create visualizations
# Plot regression predictions vs actual, RMSE comparison, and confusion matrices
print("\n" + "=" * 80)
print("VISUALIZATIONS")
print("=" * 80)

# TODO: Create figure with subplots (2x2 grid)
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle('XGBoost vs AdaBoost: Model Comparison', fontsize=14, fontweight='bold')

# TODO: Plot 1: Regression predictions vs actual (scatter plot)
# Scatter plot with both models, add diagonal line (y=x)
# Add labels, legend, grid
# axes[0, 0].scatter(...)
# axes[0, 0].plot(...)  # diagonal line

# TODO: Plot 2: RMSE comparison bar chart
# Bar chart comparing RMSE values for both models
# axes[0, 1].bar(...)

# TODO: Plot 3: XGBoost confusion matrix
# Create confusion matrix and heatmap
# cm_xgb = confusion_matrix(...)
# sns.heatmap(...)

# TODO: Plot 4: AdaBoost confusion matrix
# Create confusion matrix and heatmap
# cm_ada = confusion_matrix(...)
# sns.heatmap(...)

# TODO: Save visualization
# plt.tight_layout()
# os.makedirs('models/xgboost_adaboost', exist_ok=True)
# plt.savefig('models/xgboost_adaboost/model_comparison.png', dpi=150, bbox_inches='tight')
# plt.show()
# print("✓ Comparison plots saved")


In [ ]:
# TODO: FEATURE IMPORTANCE ANALYSIS
# Extract feature importance from all models and visualize
print("=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

# TODO: Extract feature importance from all four models
# Create DataFrames with features and importance scores, sort by importance
# xgb_reg_importance = pd.DataFrame({
#     'feature': feature_cols,
#     'importance': xgb_regressor.feature_importances_
# }).sort_values('importance', ascending=False)

# xgb_cls_importance = ...
# ada_reg_importance = ...
# ada_cls_importance = ...

# TODO: Print top features for each model
# print("\nTop 10 Features - XGBoost Regression:")
# print(xgb_reg_importance.head(10))
# print("\nTop 10 Features - XGBoost Classification:")
# print(xgb_cls_importance.head(10))
# print("\nTop 10 Features - AdaBoost Regression:")
# print(ada_reg_importance.head(10))
# print("\nTop 10 Features - AdaBoost Classification:")
# print(ada_cls_importance.head(10))

# TODO: Visualize feature importance
# Create subplots for all four models using horizontal bar charts
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle('Feature Importance Comparison', fontsize=14, fontweight='bold')

# top_n = 15  # TODO: Choose number of top features to display

# TODO: Plot XGBoost Regression feature importance
# Create horizontal bar chart with top N features
# axes[0, 0].barh(...)
# Set labels, title, grid, invert y-axis

# TODO: Plot XGBoost Classification feature importance
# axes[0, 1].barh(...)

# TODO: Plot AdaBoost Regression feature importance
# axes[1, 0].barh(...)

# TODO: Plot AdaBoost Classification feature importance
# axes[1, 1].barh(...)

# TODO: Save visualization
# plt.tight_layout()
# plt.savefig('models/xgboost_adaboost/feature_importance.png', dpi=150, bbox_inches='tight')
# plt.show()
# print("\n✓ Feature importance plots saved")


## Step 11: Save Models and Predictions

Save trained models and create prediction files for submission.


In [ ]:
# TODO: SAVE MODELS AND PREDICTIONS
# Save all trained models and create prediction DataFrames
print("=" * 80)
print("SAVING MODELS AND PREDICTIONS")
print("=" * 80)

# TODO: Create output directory
# os.makedirs('models/xgboost_adaboost', exist_ok=True)

# TODO: Save all models using joblib
# Save XGBoost regressor, XGBoost classifier, AdaBoost regressor, AdaBoost classifier
# joblib.dump(xgb_regressor, 'models/xgboost_adaboost/xgb_regression.pkl')
# joblib.dump(xgb_classifier, 'models/xgboost_adaboost/xgb_classification.pkl')
# joblib.dump(ada_regressor, 'models/xgboost_adaboost/ada_regression.pkl')
# joblib.dump(ada_classifier, 'models/xgboost_adaboost/ada_classification.pkl')

# TODO: Get prediction probabilities for classification models
# Extract probabilities for positive class (class 1) from both classifiers
# y_val_pred_xgb_cls_proba = ...
# y_test_pred_xgb_cls_proba = ...
# y_val_pred_ada_cls_proba = ...
# y_test_pred_ada_cls_proba = ...

# TODO: Extract dates from validation and test data
# Ensure dates match the filtered data (after removing NaNs)
# val_dates = ...
# test_dates = ...

# TODO: Create prediction DataFrames
# Include dates, actual values, and predictions for both models
# Include probabilities for classification
# predictions_val = pd.DataFrame({
#     'Date': val_dates,
#     'actual_heat_index': ...,
#     'predicted_heat_index_xgb': ...,
#     'predicted_heat_index_ada': ...,
#     'actual_heat_risk': ...,
#     'predicted_heat_risk_xgb': ...,
#     'predicted_heat_risk_ada': ...,
#     'predicted_heat_risk_xgb_proba': ...,
#     'predicted_heat_risk_ada_proba': ...
# })

# predictions_test = pd.DataFrame({...})

# TODO: Save predictions to CSV files
# predictions_val.to_csv('models/xgboost_adaboost/predictions_val.csv', index=False)
# predictions_test.to_csv('models/xgboost_adaboost/predictions_test.csv', index=False)

# TODO: Verify that all files were saved
# Check existence of model files and prediction files
# for file in ['xgb_regression.pkl', 'xgb_classification.pkl', 
#              'ada_regression.pkl', 'ada_classification.pkl',
#              'predictions_val.csv', 'predictions_test.csv']:
#     path = f'models/xgboost_adaboost/{file}'
#     if os.path.exists(path):
#         print(f"  ✓ {file} exists")
#     else:
#         print(f"  ✗ {file} not found!")


## Summary

After completing all sections, you should have:
- ✅ Features prepared and encoded
- ✅ XGBoost regressor and classifier trained
- ✅ AdaBoost regressor and classifier trained
- ✅ Models evaluated and compared
- ✅ Feature importance analyzed
- ✅ Models saved to `models/xgboost_adaboost/`
- ✅ Predictions saved to CSV files

**Key Differences**:
- **XGBoost**: Gradient boosting with regularization, handles missing values, generally faster
- **AdaBoost**: Adaptive boosting, simpler but can be slower
- Compare performance to determine which model performs better for your data

**Next Steps**: 
- Review `models/xgboost_adaboost/README.md` for submission requirements
- Ensure all deliverables are complete
- Proceed to `ensemble.ipynb` after all models complete their models
